<div dir="rtl">

# Z-Image-Turbo

همه‌ی سلول‌ها را به ترتیب اجرا کنید (Run all). لینک پنل در خروجی سلول لانچ چاپ می‌شود.

مدل با ComfyUI اجرا می‌شود؛ ترانسفورمر INT8 است و جمع دانلود حدود ۱۴٫۶ گیگ است.

توکن هاگینگ‌فیس (اختیاری، برای سرعت دانلود): از بخش Secrets کولب با نام HF_TOKEN بسازید.

</div>

In [ ]:
!nvidia-smi -L

In [ ]:
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login
    login(token=_tok)
    print("توکن تنظیم شد — دانلود سریع‌تر انجام می‌شود")
except Exception:
    print("توکنی در Secrets نیست — بدون توکن ادامه می‌دهیم")

In [ ]:
%%bash
pip install -q gradio
cd /content
test -d ComfyUI || git clone -q https://github.com/comfyanonymous/ComfyUI.git
cd /content/ComfyUI && pip install -q -r requirements.txt
echo ComfyUI ready

In [ ]:
%%bash
cd /content
curl -L -C - --retry 3 -o ComfyUI/models/diffusion_models/z_image_turbo_int8_convrot.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_int8_convrot.safetensors
curl -L -C - --retry 3 -o ComfyUI/models/text_encoders/qwen_3_4b.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors
curl -L -C - --retry 3 -o ComfyUI/models/vae/ae.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors
ls -la ComfyUI/models/diffusion_models/ ComfyUI/models/text_encoders/ ComfyUI/models/vae/

In [ ]:
import subprocess, time, urllib.request, os
os.chdir("/content/ComfyUI")
log = open("/tmp/comfy.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
)
print("ComfyUI در حال بالا آمدن است (pid", proc.pid, ") — منتظر جوابش می‌مانیم...", flush=True)
for i in range(90):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3) as r:
            if r.status == 200:
                print("✅ ComfyUI بالا اومد — حدود", (i + 1) * 4, "ثانیه", flush=True)
                break
    except Exception:
        pass
    if i % 5 == 0:
        print("⏳ هنوز در حال بالا آمدن...", i * 4, "ثانیه", flush=True)
    time.sleep(4)
else:
    print("❌ ComfyUI جواب نداد — لاگ زیر رو ببین")
    print(open("/tmp/comfy.log", encoding="utf-8", errors="replace").read()[-2000:])

In [ ]:
# وضعیت سرور ComfyUI — هر وقت خواستی دوباره اجرا کن
print(open("/tmp/comfy.log", encoding="utf-8", errors="replace").read()[-2000:])

In [ ]:
%%writefile /content/panel_app.py
# Z-Image-Turbo panel — Gradio UI driving a local headless ComfyUI server.
# Weights: Comfy-Org/z_image_turbo split_files (INT8 convrot transformer,
# bf16 Qwen3-4B encoder, ae VAE). Graph per the official ComfyUI example
# workflow: UNETLoader -> ModelSamplingAuraFlow shift 3.0 -> KSampler
# (steps 8, cfg 1, res_multistep, simple); CLIPLoader type "lumina2";
# negative = ConditioningZeroOut; latent = EmptySD3LatentImage.
import io
import json
import random
import time
import urllib.request
import urllib.parse
import uuid

import gradio as gr
import numpy as np
from PIL import Image

COMFY = "http://127.0.0.1:8188"
MAX_SEED = np.iinfo(np.int32).max
CLIENT_ID = uuid.uuid4().hex

RESOLUTIONS = [
    "1024x1024 (1:1)", "1152x896 (4:3-ish)", "896x1152 (3:4-ish)",
    "1280x720 (16:9)", "720x1280 (9:16)", "1344x768 (7:4)",
    "768x1344 (4:7)", "1536x640 (12:5)", "640x1536 (5:12)",
]

EXAMPLES = [
    "An action still of a man raising a cup of coffee in a busy cafe, strong rim lighting from a window, motion blur on background, candid photography, gritty urban 35mm feel, sharp facial details, natural imperfections",
    "A portrait photo of a young woman with long wavy brown hair and freckles, wearing a beige knit sweater, soft window light, shallow depth of field, photorealistic skin texture, 85mm lens",
    "A fluffy orange cat sitting on a windowsill looking at falling snow outside, warm indoor light, cozy atmosphere, photorealistic fur detail",
    "A futuristic sports car parked on a wet neon-lit Tokyo street at night, reflections on the asphalt, cinematic composition, photorealistic",
]

CSS = """
.main-title { text-align: center; margin-bottom: 5px; }
.main-title h1 { font-weight: 750; margin-bottom: 3px; font-size: 2em; }
.main-title p { color: #6b6f76; margin: 0 auto; max-width: 700px; }
.credits-line { text-align: center; font-size: 0.9em; color: #868b94; margin-top: -2px; margin-bottom: 8px; }
#generate-btn { background: linear-gradient(135deg, #2563eb, #1d4ed8); color: white; border: none; }
"""

def comfy_ready():
    try:
        with urllib.request.urlopen(COMFY + "/system_stats", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

def post_prompt(workflow):
    data = json.dumps({"prompt": workflow, "client_id": CLIENT_ID}).encode()
    req = urllib.request.Request(COMFY + "/prompt", data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["prompt_id"]

def wait_images(prompt_id, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(f"{COMFY}/history/{prompt_id}", timeout=10) as r:
                hist = json.loads(r.read())
        except Exception:
            time.sleep(2); continue
        entry = hist.get(prompt_id)
        if entry:
            status = entry.get("status", {})
            if status.get("status_str") == "error":
                msgs = status.get("messages", [])
                raise RuntimeError(f"ComfyUI error: {msgs[-1] if msgs else 'unknown'}")
            imgs = []
            for node_out in entry.get("outputs", {}).values():
                for im in node_out.get("images", []):
                    q = urllib.parse.urlencode(
                        {"filename": im["filename"], "subfolder": im.get("subfolder", ""),
                         "type": im.get("type", "output")})
                    with urllib.request.urlopen(f"{COMFY}/view?{q}", timeout=30) as r:
                        imgs.append(Image.open(io.BytesIO(r.read())).convert("RGB"))
            if imgs:
                return imgs
        time.sleep(2)
    raise TimeoutError("Generation timed out")

def parse_dims(label):
    wh = (label or "1024x1024").split("(")[0].strip()
    w, h = wh.split("x")
    return int(w), int(h)

def generate(prompt, steps, resolution, seed):
    if not comfy_ready():
        raise gr.Error("سرور ComfyUI هنوز بالا نیامده — پیام آماده‌شدن نوت‌بوک را ببین و دوباره بزن")
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("پرامپت خالی است")
    if int(seed) < 0:
        seed = random.randint(0, MAX_SEED)
    w, h = parse_dims(resolution)
    nodes = {
        "1": {"class_type": "UNETLoader",
              "inputs": {"unet_name": "z_image_turbo_int8_convrot.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "ModelSamplingAuraFlow",
              "inputs": {"model": ["1", 0], "shift": 3.0}},
        "3": {"class_type": "CLIPLoader",
              "inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "lumina2", "device": "default"}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["3", 0], "text": prompt}},
        "5": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}},
        "6": {"class_type": "VAELoader", "inputs": {"vae_name": "ae.safetensors"}},
        "7": {"class_type": "EmptySD3LatentImage",
              "inputs": {"width": w, "height": h, "batch_size": 1}},
        "8": {"class_type": "KSampler",
              "inputs": {"model": ["2", 0], "positive": ["4", 0], "negative": ["5", 0],
                         "latent_image": ["7", 0], "seed": int(seed), "steps": int(steps),
                         "cfg": 1.0, "sampler_name": "res_multistep",
                         "scheduler": "simple", "denoise": 1.0}},
        "9": {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["6", 0]}},
        "11": {"class_type": "SaveImage",
               "inputs": {"images": ["9", 0], "filename_prefix": "zimage"}},
    }
    imgs = wait_images(post_prompt(nodes))
    return imgs[0], int(seed)

with gr.Blocks(title="Z-Image-Turbo") as demo:
    gr.HTML("""
    <div class="main-title">
        <h1>Z-Image-Turbo</h1>
        <p>State-of-the-art text-to-image model by Tongyi-MAI.</p>
    </div>
    <p class="credits-line">INT8 · ComfyUI</p>
    """)
    with gr.Row():
        with gr.Column(scale=5):
            prompt = gr.Textbox(label="Prompt", lines=3,
                                placeholder="Describe the image you want to generate...")
            with gr.Row():
                steps = gr.Slider(label="Steps", minimum=1, maximum=20, value=8, step=1)
                seed = gr.Number(label="Seed", value=-1, precision=0)
            resolution = gr.Dropdown(label="Resolution", choices=RESOLUTIONS, value=RESOLUTIONS[0])
            gen_btn = gr.Button("Generate Image", elem_id="generate-btn")
        with gr.Column(scale=4):
            output = gr.Image(label="Generated Image", type="pil", height=460)
            used_seed = gr.Number(label="Seed used", precision=0)
    gr.Examples(examples=[[e] for e in EXAMPLES], inputs=[prompt], label="Example Prompts")
    gen_btn.click(fn=generate, inputs=[prompt, steps, resolution, seed], outputs=[output, used_seed])

demo.launch(share=True, css=CSS)


In [ ]:
import subprocess, time, os
os.chdir("/content")
log = open("panel.log", "w")
_p = subprocess.Popen(["python", "-u", "panel_app.py"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("پنل در پس‌زمینه استارت شد — pid:", _p.pid)
for _i in range(30):
    time.sleep(10)
    _txt = open("panel.log", encoding="utf-8", errors="replace").read()
    if "gradio.live" in _txt:
        print("✅ پنل بالا اومد")
        break
    if _p.poll() is not None:
        print("❌ پنل مُرد — خروجی لاگ رو تو سلول وضعیت ببین")
        break
    print("⏳ در حال بالا آمدن...", (_i + 1) * 10, "ثانیه")
print("—" * 40)
print(open("panel.log", encoding="utf-8", errors="replace").read()[-1500:])

In [ ]:
# وضعیت پنل — هر وقت خواستی این سلول رو دوباره اجرا کن
print(open("panel.log", encoding="utf-8", errors="replace").read()[-2000:])